In [ ]:
# import packages
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# define destination folder for the figures
OUT_DIR = os.path.join("..", "figures")
os.makedirs(OUT_DIR, exist_ok=True)

# parameters
LAG = 0
EXCLUDE_COVID = True
COVID = (2020, 2021)

# import data

# cpi data (monthly)
cpi_file = os.path.join("..", "data", "processed_data", 'cpi_data.csv')
cpi = pd.read_csv(cpi_file, delimiter=';', encoding='utf-8', index_col=[0], parse_dates=True)

# cpi data (yearly means)
cpi_yearly_file = os.path.join("..", "data", "processed_data", 'cpi_yearly.csv')
cpi_year = pd.read_csv(cpi_yearly_file, delimiter=';', encoding='utf-8', index_col=[0])

# mode data
mode_file = os.path.join("..", "data", "processed_data", 'mode_data.csv')
mode_df = pd.read_csv(
    mode_file,
    delimiter=';',
    encoding='utf-8',
    header=[0, 1],
    index_col=[0, 1],
    parse_dates=[0],
    date_format='%Y-%m-%d'
)

# mode data with short column names, year as integer index
mode_short_file = os.path.join("..", "data", "processed_data", 'mode_short.csv')
mode = pd.read_csv(mode_short_file, delimiter=';', encoding='utf-8', header=[0, 1], index_col=[0, 1])
mode.index.names = ['year', 'purpose']
mode.columns.names = ['mode', 'indicator']

# income data
income_file = os.path.join("..", "data", "processed_data", 'income_data.csv')
income_df = pd.read_csv(income_file, delimiter=';', encoding='utf-8', index_col=[0], parse_dates=True)

# fuel weights
weights_file = os.path.join("..", "data", "processed_data", 'fuel_weights.csv')
weights_df = pd.read_csv(weights_file, delimiter=';', encoding='utf-8', index_col=[0], parse_dates=True)

# project dataset
project_file = os.path.join("..", "data", "processed_data", 'project_dataset.csv')
df = pd.read_csv(project_file, delimiter=';', encoding='utf-8', index_col=0)

def get(m: str, indicator: str, purpose: str ="Total") -> pd.Series:
  return mode.xs(purpose, level="purpose")[(m, indicator)]


# plot settings
COL = {
    "train": "#E07A2F",
    "car": "#1F3A5F",
    "diesel": "#6B7280",
    "petrol": "#6C8EBF",
    "lpg": "#9CA3AF",
    "ratio": "#B4432F",
}
plt.rcParams.update({
    "figure.figsize": (9, 5.2),
    "figure.dpi": 110,
    "savefig.dpi": 200,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.titleweight": "bold",
    "axes.titlesize": 14,
    "axes.titlelocation": "left",
})

def save(fig: plt.Figure, name: str) -> None:
  """Export figure to file."""
  fig.tight_layout()
  fig.savefig(os.path.join(OUT_DIR, f"{name}.png"), bbox_inches="tight")


def covid_band(ax: plt.Axes, label: bool = True) -> None:
  """Add indication for COVID period."""
  ax.axvspan(COVID[0] - 0.5, COVID[1] + 0.5, color="#9CA3AF", alpha=0.15, lw=0)
  if label:
    ax.text(sum(COVID) / 2, ax.get_ylim()[1],"COVID-19", ha="center", va="top",fontsize=9, color="#6B7280",)

In [ ]:

# graph 1
fig, ax = plt.subplots()
ax.plot(cpi.index, cpi["CPI_Train"], color=COL["train"], lw=2.8, label="Train fare")
ax.plot(cpi.index, cpi["CPI_Petrol"], color=COL["petrol"], lw=1.6, label="Petrol")
ax.plot(cpi.index, cpi["CPI_Diesel"], color=COL["diesel"], lw=1.6, label="Diesel")
ax.plot(cpi.index, cpi["CPI_LPG"], color=COL["lpg"], lw=1.4, label="LPG")
ax.axhline(100, color="#9CA3AF", lw=1, ls="--")
ax.set_title("Consumer price index: train fare vs fuel (2015 = 100)")
ax.set_ylabel("CPI")
ax.legend(frameon=False, ncol=4, loc="upper left")
save(fig, "1_cpi_train_vs_fuel")
plt.show()


In [ ]:
# graph 2
START, END = 1999, 2025
cpi_m = cpi.loc[f"{START}-01-01":f"{END}-12-31"]
cpi_y = cpi_year.loc[START:END]

fig, ax = plt.subplots()
ax.plot(cpi_m.index, cpi_m["Ratio"], color=COL["ratio"], lw=1, alpha=0.35, label="Monthly")
ax.plot(pd.to_datetime(cpi_y.index.astype(str) + "-07-01"), cpi_y["Ratio"], color=COL["ratio"], lw=2.8, label="Annual mean")

top, bottom = cpi_y["Ratio"].idxmax(), cpi_y["Ratio"].idxmin()
for y, offset in ((top, 10), (bottom, -16)):
    ax.annotate(
        f"{y}: {cpi_y.loc[y, 'Ratio']:.2f}",
        (pd.Timestamp(f"{y}-07-01"),
         cpi_y.loc[y, "Ratio"]),
        textcoords="offset points",
        xytext=(0, offset), ha="center",
        fontsize=9
    )
    
ax.axhline(1, color="#9CA3AF", lw=1, ls="--")
ax.set_xlim(pd.Timestamp(f"{START}-01-01"), pd.Timestamp(f"{END}-12-31"))
ax.set_title("Fare-to-fuel price ratio (higher = train relatively more expensive)")
ax.set_ylabel("Train CPI / fuel CPI")
ax.legend(frameon=False)
save(fig, "2_fare_fuel_ratio")
plt.show()

In [ ]:
# graph 3
fig, ax = plt.subplots()
share_car = df["car_share"]
ax.stackplot(df.index, share_car, df["train_share"], labels=["Car", "Train"], colors=[COL["car"], COL["train"]], alpha=0.9)
ax.set_ylim(0, 100)
ax.set_xlim(df.index.min(), df.index.max())
ax.set_ylabel("% of car + train kilometres")
ax.set_title("Modal split car vs train (share of distance travelled)")
ax.legend(loc="lower left", frameon=False)
for y in (df.index.min(), df.index.max()):
    ax.text(y, share_car[y] + 1.2, f"{df.loc[y, 'train_share']:.1f}% train", color="white", fontsize=9, ha="left" if y == df.index.min() else "right")
save(fig, "3_modal_split")
plt.show()


In [ ]:

# graph 4
fig, ax1 = plt.subplots()
ax1.plot(df.index, df["Ratio"], color=COL["ratio"], lw=2.6, label="Fare-to-fuel ratio")
ax1.set_ylabel("Fare-to-fuel ratio", color=COL["ratio"])
ax2 = ax1.twinx()
ax2.plot(df.index, df["train_share"], color=COL["train"], lw=2.6, label="Train share of km")
ax2.set_ylabel("Train share of car + train km (%)", color=COL["train"])
ax2.grid(False)
ax2.spines["right"].set_visible(True)
covid_band(ax1)
ax1.set_title("Price ratio and train share over time")
save(fig, "4_ratio_en_treinaandeel")
plt.show()


In [ ]:
# graph 5
d = df.dropna(subset=["Ratio", "train_share"])
fig, ax = plt.subplots(figsize=(8.5, 5.6))
sc = ax.scatter(d["Ratio"], d["train_share"], c=d.index, cmap="viridis", s=70, edgecolor="white", linewidth=1, zorder=3)
cb = fig.colorbar(sc, ax=ax, pad=0.02)
cb.set_label("Year")
fit = d.drop(index=[y for y in range(COVID[0], COVID[1] + 1) if y in d.index]) if EXCLUDE_COVID else d
k, b = np.polyfit(fit["Ratio"], fit["train_share"], 1)
xs = np.linspace(d["Ratio"].min(), d["Ratio"].max(), 50)
ax.plot(xs, k * xs + b, color=COL["ratio"], lw=2, ls="--", label=f"Linear fit{' (excl. COVID)' if EXCLUDE_COVID else ''}: slope {k:.1f} pp per ratio point")
for y in (d.index.min(), d.index.max(), *COVID):
    if y in d.index:
        ax.annotate(str(y), (d.loc[y, "Ratio"], d.loc[y, "train_share"]), textcoords="offset points", xytext=(6, 6), fontsize=9)
ax.set_xlabel("Fare-to-fuel ratio")
ax.set_ylabel("Train share of car + train km (%)")
ax.set_title("Does a higher relative train price lower the train share?")
ax.legend(frameon=False, loc="best")
save(fig, "5_scatter_ratio_vs_share")
plt.show()


In [ ]:
# elasticity analysis
def elasticity(y, x, lag=LAG, exclude_covid=EXCLUDE_COVID):
    """Helling van ln(y) op ln(x). x wordt 'lag' jaar vertraagd. Geeft (e, r2, n, p_approx)."""
    t = pd.concat([y, x.shift(lag)], axis=1, keys=["y", "x"]).dropna()
    if exclude_covid:
        t = t.drop(index=[j for j in range(COVID[0], COVID[1] + 1) if j in t.index])
    t = t[(t > 0).all(axis=1)]
    lx, ly = np.log(t["x"]), np.log(t["y"])
    e, c = np.polyfit(lx, ly, 1)
    resid = ly - (e * lx + c)
    r2 = 1 - resid.var() / ly.var()
    n = len(t)
    se = np.sqrt(resid.var(ddof=2) / ((lx - lx.mean()) ** 2).sum())
    return {"elasticity": e, "se": se, "r2": r2, "n": n, "intercept": c, "x": lx, "y": ly}

res = {name: elasticity(df[col], df["Ratio"]) for name, col in [("Train km", "train_km"), ("Car km", "car_km"), ("Train share", "train_share")]}
tabel = pd.DataFrame({k: {"elasticity": v["elasticity"], "std. error": v["se"], "R2": v["r2"], "n": v["n"]} for k, v in res.items()}).T


In [ ]:
# graph 6
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
for ax, name, color in zip(axes, ["Train km", "Car km"], [COL["train"], COL["car"]]):
    r = res[name]
    ax.scatter(r["x"], r["y"], color=color, s=55, edgecolor="white", zorder=3)
    xs = np.linspace(r["x"].min(), r["x"].max(), 50)
    ax.plot(xs, r["elasticity"] * xs + r["intercept"], color="black", lw=1.8, ls="--")
    ax.set_title(f"{name}: elasticity {r['elasticity']:.2f} (R² {r['r2']:.2f})", fontsize=12)
    ax.set_xlabel("ln(fare-to-fuel ratio)")
    ax.set_ylabel(f"ln({name.lower()} per person per day)")
save(fig, "6_loglog_elasticiteit")
plt.show()

In [ ]:
# graph 7
purposes = [p for p in mode.index.get_level_values("purpose").unique()]
rows = []
for p in purposes:
    r = elasticity(get("Train", "km_day", p), df["Ratio"])
    rows.append((p, r["elasticity"], 1.96 * r["se"]))
pur = pd.DataFrame(rows, columns=["purpose", "e", "ci"]).sort_values("e")

fig, ax = plt.subplots()
ax.barh(pur["purpose"], pur["e"], xerr=pur["ci"], color=COL["train"], alpha=0.9, error_kw={"ecolor": "#4B5563", "capsize": 4})
ax.axvline(0, color="black", lw=1)
ax.set_xlabel("Elasticity of train km w.r.t. fare-to-fuel ratio (± 95% CI, rough)")
ax.set_title("Price sensitivity of train use by travel purpose")
ax.grid(axis="y", visible=False)
save(fig, "7_elasticiteit_per_reismotief")
plt.show()



In [ ]:
# elasticity with control variables (multiple log-log regression)

# real income per capita and COVID-19
df_ctrl = df.dropna(subset=["real_income"])

def elasticity_controls(y, controls=(), data=df_ctrl):
    """Log-log OLS of ln(y) on ln(Ratio) plus control variables.
    Controls starting with 'ln_' are logged, others (e.g. 'covid') are used as they are."""
    cols = [y, "Ratio"] + [c.removeprefix("ln_") for c in controls]
    t = data[list(dict.fromkeys(cols))].dropna()
    t = t[(t[[y, "Ratio"]] > 0).all(axis=1)]
    X = [np.ones(len(t)), np.log(t["Ratio"])]
    for c in controls:
        X.append(np.log(t[c.removeprefix("ln_")]) if c.startswith("ln_") else t[c])
    X = np.column_stack(X)
    ly = np.log(t[y]).values
    beta, *_ = np.linalg.lstsq(X, ly, rcond=None)
    resid = ly - X @ beta
    n, k = X.shape
    sigma2 = resid @ resid / (n - k)
    se = np.sqrt(np.diag(sigma2 * np.linalg.inv(X.T @ X)))
    r2 = 1 - resid.var() / ly.var()
    return {"elasticity": beta[1], "std. error": se[1], "R2": r2, "n": n}

specs = {
    "1. Ratio only": [],
    "2. + income": ["ln_real_income"],
    "3. + income + COVID": ["ln_real_income", "covid"],
    "4. + income + COVID (2020–22)": ["ln_real_income", "covid_ext"],
}
rows = []
for col, label in [("train_km", "Train km"), ("car_km", "Car km"), ("train_share", "Train share")]:
    for name, controls in specs.items():
        rows.append({"dependent": label, "model": name, **elasticity_controls(col, controls)})

tabel_ctrl = pd.DataFrame(rows).set_index(["dependent", "model"])
tabel_ctrl["95% CI"] = 1.96 * tabel_ctrl["std. error"]
tabel_ctrl.round(3)


In [ ]:
# graph 8
fig, ax = plt.subplots()
models = list(specs)
y_pos = np.arange(len(models))
for i, (label, color) in enumerate([("Train km", COL["train"]), ("Car km", COL["car"]), ("Train share", COL["ratio"])]):
    t = tabel_ctrl.loc[label]
    ax.barh(y_pos + (i - 1) * 0.27, t["elasticity"], height=0.25, xerr=t["95% CI"], color=color, label=label, error_kw={"ecolor": "#4B5563", "capsize": 3})
ax.axvline(0, color="black", lw=1)
ax.set_yticks(y_pos, models)
ax.invert_yaxis()
ax.set_xlabel("Elasticity w.r.t. fare-to-fuel ratio (± 95% CI)")
ax.set_title("Elasticity with and without control variables")
ax.legend(frameon=False)
save(fig, "8_elasticiteit_met_controls")
plt.show()